# Daily Household Transactions

LOAD DATASET

In [1]:
import pandas as pd
import numpy as np

df4 = pd.read_csv("../Datasets/Daily Household Transactions.csv")
df4.head()

,Date,Mode,Category,Subcategory,Note,Amount,Income/Expense,Currency
0,20/09/2018 12:04:08,Cash,Transportation,Train,2 Place 5 to Place 0,30.0,Expense,INR
1,20/09/2018 12:03:15,Cash,Food,snacks,Idli medu Vada mix 2 plates,60.0,Expense,INR
2,19/09/2018,Saving Bank account 1,subscription,Netflix,1 month subscription,199.0,Expense,INR
3,17/09/2018 23:41:17,Saving Bank account 1,subscription,Mobile Service Provider,Data booster pack,19.0,Expense,INR
4,16/09/2018 17:15:08,Cash,Festivals,Ganesh Pujan,Ganesh idol,251.0,Expense,INR


Check Information 

In [2]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2461 entries, 0 to 2460
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            2461 non-null   object 
 1   Mode            2461 non-null   object 
 2   Category        2461 non-null   object 
 3   Subcategory     1826 non-null   object 
 4   Note            1940 non-null   object 
 5   Amount          2461 non-null   float64
 6   Income/Expense  2461 non-null   object 
 7   Currency        2461 non-null   object 
dtypes: float64(1), object(7)
memory usage: 153.9+ KB


REMOVE DUPLICATES

In [3]:
df4.duplicated().sum()

np.int64(9)

In [4]:
df4 = df4.drop_duplicates()

STANDARDIZE SCHEMA

In [5]:
df4 = df4.rename(columns={
    "Note": "Transaction Description",
    "Subcategory": "Merchant",
    "Mode": "Account Name",
    "Income/Expense": "Type"
})

CLEAN DATE COLUMN

In [6]:
# --- STEP 1: Clean raw text ---
df4["Date"] = df4["Date"].astype(str).str.strip()

# --- STEP 2: Try parsing dates (day-first format) ---
df4["Date"] = pd.to_datetime(df4["Date"], errors="coerce", dayfirst=True)

# --- STEP 3: Count invalid dates ---
invalid_dates = df4["Date"].isna().sum()
print("[INFO] Invalid date entries:", invalid_dates)

# --- STEP 4: Neighbour-based imputation (ffill + bfill) ---
if invalid_dates > 0:
    print("[INFO] Performing neighbour-based date imputation...")
    
    # Forward fill first (use previous valid date)
    df4["Date"] = df4["Date"].fillna(method="ffill")
    
    # Backward fill leftover NA (top rows)
    df4["Date"] = df4["Date"].fillna(method="bfill")

# --- STEP 5: Convert to date-only format (optional) ---
df4["Date"] = df4["Date"].dt.date

print("[INFO] Final missing dates:", df4["Date"].isna().sum())

[INFO] Invalid date entries: 1149
[INFO] Performing neighbour-based date imputation...
[INFO] Final missing dates: 0


C:\Users\Geemeth\AppData\Local\Temp\ipykernel_16672\2733025168.py:16: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df4["Date"] = df4["Date"].fillna(method="ffill")
C:\Users\Geemeth\AppData\Local\Temp\ipykernel_16672\2733025168.py:19: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df4["Date"] = df4["Date"].fillna(method="bfill")


CLEAN AMOUNT COLUMN

In [7]:
df4["Amount"] = pd.to_numeric(df4["Amount"], errors="coerce")
df4 = df4[df4["Amount"] > 0]

USER ID

In [8]:
# No user field, but values follow 1-person lifestyle.
df4["UserID"] = "US009"

CREATE TransactionID

In [9]:
df4["TransactionID"] = [f"DHT{idx:06d}" for idx in range(1, len(df4) + 1)]

FIX TYPE COLUMN

In [10]:
# Convert to Title case
df4["Type"] = df4["Type"].astype(str).str.strip().str.title()

In [11]:
# Convert Transfer-Out → Expense (standardization)
df4.loc[df4["Type"].str.contains("Transfer", case=False), "Type"] = "Expense"

print("[INFO] Type Distribution:")
print(df4["Type"].value_counts())

[INFO] Type Distribution:
Type
Expense    2327
Income      125
Name: count, dtype: int64


FIX CATEGORY NAMES

In [12]:
df4["Category"] = df4["Category"].astype(str).str.strip().str.title()

MERCHANT COLUMN

In [13]:
df4["Merchant"] = df4["Merchant"].astype(str).str.title()

ACCOUNT NAME

In [14]:
# Mode = Cash / Bank Account / GPay etc.
df4["Account Name"] = df4["Account Name"].astype(str).str.title()

In [15]:
df4.head()

,Date,Account Name,Category,Merchant,Transaction Description,Amount,Type,Currency,UserID,TransactionID
0,2018-09-20,Cash,Transportation,Train,2 Place 5 to Place 0,30.0,Expense,INR,US009,DHT000001
1,2018-09-20,Cash,Food,Snacks,Idli medu Vada mix 2 plates,60.0,Expense,INR,US009,DHT000002
2,2018-09-20,Saving Bank Account 1,Subscription,Netflix,1 month subscription,199.0,Expense,INR,US009,DHT000003
3,2018-09-17,Saving Bank Account 1,Subscription,Mobile Service Provider,Data booster pack,19.0,Expense,INR,US009,DHT000004
4,2018-09-16,Cash,Festivals,Ganesh Pujan,Ganesh idol,251.0,Expense,INR,US009,DHT000005


VALIDATE REQUIRED COLUMNS

In [16]:
required = ["TransactionID","UserID","Date","Category","Amount","Type"]
df4[required].isna().sum()

TransactionID    0
UserID           0
Date             0
Category         0
Amount           0
Type             0
dtype: int64

In [17]:
# Total number of transactions
print("Total Transactions:", len(df4))
print("\nTransactions per User:")
print(df4["UserID"].value_counts())

Total Transactions: 2452

Transactions per User:
UserID
US009    2452
Name: count, dtype: int64


SAVE CLEANED DATASET

In [18]:
import os
os.makedirs("Tofinal", exist_ok=True)

df4.to_csv("Tofinal/Daily Household Transactions.csv", index=False)
print("File saved successfully!")

File saved successfully!
